# 11 — Orientierung und Trainingsmatrix

Zwei Fragen, die alle bisherigen Notebooks offen lassen:

**Teil A — Orientierung.** In Notebook 10 unterscheiden sich linke und
rechte Körperseite bei identischen Stürzen um bis zu Faktor neun. Ist
das die Achsenorientierung? Prüfung: Die linke Seite wird gespiegelt
bzw. gedreht und erneut bewertet. Schließt eine Spiegelung die Lücke
zur rechten Seite, ist die Orientierung die Ursache.

**Teil B — Trainingsmatrix.** Welche Trainingskette ist für welche
Zielposition die richtige? Statt aller denkbaren Kombinationen (viele
davon ohne Stürze im Testsatz und damit nicht bewertbar) werden die
fünf Ketten verglichen, die eine Entscheidung tragen:

| Kette | Training | beantwortet |
|---|---|---|
| W  | SisFall (Hüfte) | Referenz |
| WK | W → KFall | Zwischenstufe allein |
| L0 | Bein + Fuß **ohne** Vortraining | Braucht man die Rumpfdaten überhaupt? |
| B2 | WK → Bein + Fuß | bisheriges Lower-Limb-Modell (Notebook 10) |
| B2r | wie B2, mit Rotations-/Spiegelaugmentierung | Behebt Augmentierung die Orientierungsabhängigkeit? |

Bein = CAPPIMU-Schienbein (Stürze), Fuß = ImbalanSense (ohne Stürze).
Bewertet auf: CAPPIMU-Schienbein (5-fach CV, original und zufällig
gedreht), ImbalanSense (CV), Smart-Insole (extern), Pilotdaten an
Hüfte und beiden Knöcheln (extern, eigene Hardware).

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q scipy scikit-learn tensorflow openpyxl

import os, sys, shutil, time, json, pickle, zipfile
from pathlib import Path
import numpy as np
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

# fall_lib laden: zuerst GitHub, sonst Kopie auf Drive. Schlaegt beides
# fehl, wird abgebrochen -- ohne fall_lib waeren alle Ergebnisse falsch
# (andere Fensterlaenge, anderes Heruntertasten).
import urllib.request, shutil as _sh
_LIB_URL = "https://raw.githubusercontent.com/Alyelb/ML-Modell-for-Fall-detection/main/fall_lib.py"
_LIB_DRIVE = "/content/drive/MyDrive/ML_Fall_Detection/fall_lib.py"
try:
    urllib.request.urlretrieve(_LIB_URL, "/content/fall_lib.py")
    print("[fall_lib] von GitHub geladen")
except Exception as _e:
    print(f"[fall_lib] GitHub nicht erreichbar ({_e}) -- nutze Drive-Kopie")
    _sh.copy(_LIB_DRIVE, "/content/fall_lib.py")
sys.path.insert(0, "/content")
import fall_lib as fl
import importlib; importlib.reload(fl)
# Pflichtfunktionen pruefen
for _f in ("new_run", "build_windows", "znorm", "WIN_LEN"):
    assert hasattr(fl, _f), f"fall_lib unvollstaendig ({_f} fehlt) -- Datei pruefen."
print("[fall_lib] Fensterlaenge", fl.WIN_LEN, "bei", fl.FS_TARGET, "Hz")
assert hasattr(fl, "augment_orientation"), "fall_lib.py auf GitHub ist veraltet."

BASE  = "/content/drive/MyDrive/ML_Fall_Detection"
fl.RESULTS_ROOT = f"{BASE}/results"
MODEL = f"{BASE}/results/fall_model.keras"
PILOT = f"{BASE}/Pilotdaten"
CACHE = Path(f"{BASE}/cache"); CACHE.mkdir(exist_ok=True)

def hole(name, ziel):
    src = f"{BASE}/{name}"
    if Path(ziel).exists():
        return ziel
    if not Path(src).exists():
        raise FileNotFoundError(f"'{name}' nicht gefunden unter {BASE}")
    shutil.copytree(src, ziel)
    return ziel

def cached(name, fn):
    p = CACHE / f"{name}.pkl"
    if p.exists():
        print(f"[cache] {name}")
        return pickle.load(open(p, "rb"))
    r = fn(); pickle.dump(r, open(p, "wb")); return r

from tensorflow import keras
def lade_modell(pfad):
    m = keras.models.load_model(pfad, compile=False)
    m.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return m

basis = lade_modell(MODEL)
run = fl.new_run("11_orientierung_matrix")
ergebnisse = {}

## Daten

Rohfenster (g, °/s) werden unnormiert gehalten, weil Drehung und
Spiegelung **vor** der Z-Normalisierung erfolgen müssen. Normiert wird
erst unmittelbar vor der Vorhersage bzw. dem Training.

In [ ]:
CAP_ROOT = next(Path(f"{BASE}/CAPPIMU Dataset").rglob("subject_1")).parent

def roh(trials):
    X, y, g, a = fl.build_windows(trials, mode="standard")
    return X, y, g, a

def vorhersage(m, X, bs=2048):
    out = []
    for i in range(0, len(X), bs):
        out.append(m.predict(fl.znorm(X[i:i+bs]), batch_size=512, verbose=0).ravel())
    return np.concatenate(out) if out else np.empty(0)

def kennzahlen(p, y, thr=0.5):
    tp = int(((p > thr) & (y == 1)).sum()); fn = int(((p <= thr) & (y == 1)).sum())
    fp = int(((p > thr) & (y == 0)).sum()); tn = int(((p <= thr) & (y == 0)).sum())
    h = (fp + tn) * fl.WIN_SEC * (1 - fl.OVERLAP) / 3600
    return {"Se": round(tp / (tp + fn), 4) if tp + fn else None,
            "Sp": round(tn / (tn + fp), 4) if tn + fp else None,
            "fp_per_hour": round(fp / h, 1) if h else None,
            "TP": tp, "FN": fn, "FP": fp, "TN": tn}

# Pilotdaten (eigene Hardware)
pilot = [t for t in fl.load_pilot(PILOT) if "INSOLE" not in t["position"]]   # Sohle ausgeschlossen
PIL = {}
for pos in sorted({t["position"] for t in pilot}):
    PIL[pos] = roh([t for t in pilot if t["position"] == pos])

## Teil A — Orientierung (Basismodell)

Für jedes Paar linker/rechter Sensoren wird die linke Seite an jeder
der drei Achsenebenen gespiegelt; zusätzlich 20 zufällige Drehungen.
Die Spannweite über die Drehungen zeigt, wie stark das Ergebnis allein
von der Einbaulage abhängt. Die Betragsmerkmale des Random Forest
dienen als Kontrolle: Sie müssen unverändert bleiben.

In [ ]:
N_ROT = 20
ROT = fl.random_rotation_matrices(N_ROT, seed=7)
I3 = np.eye(3)

def orientierung(m, X, y, name):
    r = {"original": kennzahlen(vorhersage(m, X), y)}
    for k in range(3):
        r[f"spiegel_{'xyz'[k]}"] = kennzahlen(vorhersage(m, fl.rotate_windows(X, I3, k)), y)
    se = [kennzahlen(vorhersage(m, fl.rotate_windows(X, R)), y)["Se"] for R in ROT]
    r["rot_Se_mean"] = float(np.mean(se)); r["rot_Se_min"] = float(np.min(se))
    r["rot_Se_max"] = float(np.max(se))
    sp = " ".join(f"{k[-1]}:{r[k]['Se']:.2f}" for k in r if k.startswith("spiegel"))
    print(f"  {name:<22} Se orig {r['original']['Se']:.3f} | gespiegelt {sp} | "
          f"Drehung {r['rot_Se_mean']:.2f} [{r['rot_Se_min']:.2f}; {r['rot_Se_max']:.2f}]")
    return r

teilA = {}
for paar in [("arm_l", "arm_r"), ("wrist_l", "wrist_r"), ("knee_l", "knee_r")]:
    for pos in paar:
        tr = cached(f"cappimu_{pos}", lambda: fl.load_cappimu(CAP_ROOT, pos))
        if not tr:
            continue
        X, y, g, a = roh(tr)
        f = y == 1                                   # Orientierung betrifft v. a. Se
        teilA[f"CAPPIMU {pos}"] = orientierung(basis, X[f], y[f], f"CAPPIMU {pos}")
        del X
for pos, (X, y, g, a) in PIL.items():
    if y.sum():
        teilA[f"Pilot {pos}"] = orientierung(basis, X[y == 1], y[y == 1], f"Pilot {pos}")

# Kontrolle Random-Forest-Merkmale
X, y, _, _ = PIL[next(iter(PIL))]
d = np.abs(fl.extract_features(X[:200]) - fl.extract_features(fl.rotate_windows(X[:200], ROT[0], 0)))
print(f"\nBetragsmerkmale nach Drehung+Spiegelung: max. rel. Abweichung "
      f"{(d.max(0) / (np.abs(fl.extract_features(X[:200])).max(0) + 1e-9)).max():.1e}")
ergebnisse["teilA_orientierung"] = teilA

**Lesart:** Erreicht die gespiegelte linke Seite das Niveau der rechten,
erklärt die Orientierung die Seitenlücke. Liegt die Spannweite über
zufällige Drehungen weit auseinander, ist das CNN für eine Sohle mit
wechselnder Einbaulage ohne Gegenmaßnahme nicht geeignet.

## Teil B — Trainingsmatrix

Fünffache, subjektunabhängige Kreuzvalidierung über CAPPIMU-Schienbein
und ImbalanSense wie in Notebook 10. In jeder Falte werden L0, B2 und
B2r neu trainiert; W und WK sind faltenunabhängig.

B2 und B2r werden hier **vollständig** nachtrainiert (), anders
als in Notebook 10 (erste Faltungsschicht eingefroren). Eine
Drehinvarianz muss bereits in der ersten Schicht entstehen; mit
eingefrorener Schicht wäre der Vergleich zugunsten von B2 verzerrt.
Der Unterschied zwischen B2 und B2r ist damit allein die
Augmentierung.

In [ ]:
N_FOLDS = 5
MAX_ADL_CAP, MAX_ADL_IMB = 20000, 10000
rng = np.random.default_rng(42)

knie = cached("cappimu_knee_l", lambda: fl.load_cappimu(CAP_ROOT, "knee_l")) + \
       cached("cappimu_knee_r", lambda: fl.load_cappimu(CAP_ROOT, "knee_r"))
Xc, yc, gc, ac = roh(knie); del knie
def aus_cache(name):
    p = CACHE / f"{name}.pkl"
    if not p.exists():
        raise FileNotFoundError(f"{p} fehlt -- bitte zuerst Notebook 10 ausfuehren.")
    return pickle.load(open(p, "rb"))

imb = aus_cache("imbalansense")
Xi, yi, gi, ai = roh(imb); del imb
si = aus_cache("smart_insole")
Xs, ys, gs, as_ = roh(si); del si
print(f"Schienbein {len(yc)} Fenster ({int(yc.sum())} Stuerze), "
      f"ImbalanSense {len(yi)}, Smart-Insole {len(ys)}")

# WK: einmal berechnen und sichern
WK_PFAD = f"{BASE}/results/fall_model_kfall.keras"
if Path(WK_PFAD).exists():
    m_wk = lade_modell(WK_PFAD)
else:
    kf = fl.load_kfall(hole("KFALL/sensor_data_new", "/content/KFall"),
                       label_dir=hole("KFALL/label_data_new", "/content/KFall_labels"))
    Xk, yk, _, _ = fl.build_windows(kf, mode="standard"); del kf
    m_wk = fl.finetune(basis, fl.znorm(Xk), yk); del Xk
    m_wk.save(WK_PFAD)

def kappe(idx, y, n):
    f = idx[y[idx] == 1]; a = idx[y[idx] == 0]
    return np.r_[f, rng.choice(a, n, replace=False) if len(a) > n else a]

def trainingssatz(tr_c, tr_i, aug=False):
    i_c = kappe(tr_c, yc, MAX_ADL_CAP)
    i_i = rng.choice(tr_i, min(MAX_ADL_IMB, len(tr_i)), replace=False)
    X = np.concatenate([Xc[i_c], Xi[i_i]]); y = np.concatenate([yc[i_c], yi[i_i]])
    if aug:
        X, y = fl.augment_orientation(X, y, copies=1, seed=int(rng.integers(1e6)))
    return fl.znorm(X), y

def von_null(X, y):
    m = fl.build_cnn()
    return fl.train(m, X, y)

In [ ]:
sp_c = fl.group_kfold_splits(gc, yc, n_splits=N_FOLDS)
sp_i = fl.group_kfold_splits(gi, None, n_splits=N_FOLDS)
KETTEN = ["W", "WK", "L0", "B2", "B2r"]
falten = {k: [] for k in KETTEN}

def bewerte(m, te_c, te_i, k_rot):
    r = {"knie": kennzahlen(vorhersage(m, Xc[te_c]), yc[te_c]),
         "imb":  kennzahlen(vorhersage(m, Xi[te_i]), yi[te_i]),
         "si":   kennzahlen(vorhersage(m, Xs), ys)}
    f = te_c[yc[te_c] == 1]
    se = [kennzahlen(vorhersage(m, fl.rotate_windows(Xc[f], ROT[j])), yc[f])["Se"]
          for j in k_rot]
    r["knie_rot_Se"] = float(np.mean(se))
    for pos, (X, y, _, _) in PIL.items():
        r[f"pilot_{pos}"] = kennzahlen(vorhersage(m, X), y)
    return r

for i, ((tr_c, te_c), (tr_i, te_i)) in enumerate(zip(sp_c, sp_i), 1):
    t = time.time()
    k_rot = range(5)                                  # 5 Drehungen je Falte
    print(f"\nFalte {i}/{N_FOLDS}")
    falten["W"].append(bewerte(basis, te_c, te_i, k_rot))
    falten["WK"].append(bewerte(m_wk, te_c, te_i, k_rot))
    X, y = trainingssatz(tr_c, tr_i)
    m = von_null(X, y);           falten["L0"].append(bewerte(m, te_c, te_i, k_rot)); del m
    m = fl.finetune(m_wk, X, y, freeze=0);  falten["B2"].append(bewerte(m, te_c, te_i, k_rot)); del m
    X, y = trainingssatz(tr_c, tr_i, aug=True)
    m = fl.finetune(m_wk, X, y, freeze=0);  falten["B2r"].append(bewerte(m, te_c, te_i, k_rot)); del m, X
    for k in KETTEN:
        r = falten[k][-1]
        print(f"  {k:<4} Knie Se {r['knie']['Se']:.3f} (gedreht {r['knie_rot_Se']:.3f})  "
              f"IMB FP/h {r['imb']['fp_per_hour']:>6}  SI FP/h {r['si']['fp_per_hour']:>6}")
    print(f"  ({time.time()-t:.0f}s)")

## Ergebnis

In [ ]:
def pool(liste, schl):
    tp = sum(r[schl]["TP"] for r in liste); fn = sum(r[schl]["FN"] for r in liste)
    fp = sum(r[schl]["FP"] for r in liste); tn = sum(r[schl]["TN"] for r in liste)
    h = (fp + tn) * fl.WIN_SEC * (1 - fl.OVERLAP) / 3600
    return {"Se": tp / (tp + fn) if tp + fn else None,
            "Se_ci95": fl.wilson_ci(tp, tp + fn) if tp + fn else None,
            "fp_per_hour": fp / h if h else None}

pos_pilot = list(PIL)
kopf = f"{'Kette':<5}{'Knie Se':>9}{'gedreht':>9}{'IMB FP/h':>10}{'SI FP/h':>9}"
for p in pos_pilot:
    kopf += f"{p[:10]+' Se':>14}{'FP/h':>7}"
print(kopf); print("-" * len(kopf))
tabelle = {}
for k in KETTEN:
    L = falten[k]
    z = {"knie": pool(L, "knie"), "knie_rot_Se": float(np.mean([r["knie_rot_Se"] for r in L])),
         "imb": pool(L, "imb"), "si": pool(L, "si")}
    zeile = (f"{k:<5}{z['knie']['Se']:>9.3f}{z['knie_rot_Se']:>9.3f}"
             f"{z['imb']['fp_per_hour']:>10.1f}{z['si']['fp_per_hour']:>9.1f}")
    for p in pos_pilot:
        z[p] = pool(L, f"pilot_{p}")
        se = z[p]["Se"]
        zeile += f"{(f'{se:.2f}' if se is not None else '-'):>14}{z[p]['fp_per_hour']:>7.0f}"
    print(zeile); tabelle[k] = z
ergebnisse["teilB_matrix"] = {"tabelle": tabelle, "falten": falten}

## Gepaarte Vergleiche

* **B2 vs L0** — Lohnt das Vortraining auf Rumpfdaten?
* **B2r vs B2** — Kostet die Augmentierung Sensitivität, und gewinnt sie
  Robustheit gegenüber Drehung?

In [ ]:
vgl = {}
for a, b, schl, met in [("L0", "B2", "knie", "Se"), ("L0", "B2", "imb", "Sp"),
                        ("B2", "B2r", "knie", "Se"), ("B2", "B2r", "knie_rot_Se", "Se"),
                        ("B2", "B2r", "imb", "Sp"), ("B2", "B2r", "si", "Sp")]:
    print(f"\n{b} vs {a} — {schl}")
    A = [{"Se": r[schl]} if schl == "knie_rot_Se" else r[schl] for r in falten[a]]
    B = [{"Se": r[schl]} if schl == "knie_rot_Se" else r[schl] for r in falten[b]]
    vgl[f"{b}_vs_{a}_{schl}_{met}"] = fl.paired_comparison(A, B, a, b, metric=met)
ergebnisse["vergleiche"] = vgl

## Endmodell mit Augmentierung

In [ ]:
alle_c = np.arange(len(yc)); alle_i = np.arange(len(yi))
X, y = trainingssatz(alle_c, alle_i, aug=True)
m_final = fl.finetune(m_wk, X, y, freeze=0); del X
pfad = f"{BASE}/results/fall_model_lowerlimb_aug.keras"
m_final.save(pfad); print("gesichert:", pfad)

fl.save_results(run, ergebnisse,
    config={**fl.default_config(), "n_folds": N_FOLDS, "n_rot_teilA": N_ROT,
            "max_adl_cap": MAX_ADL_CAP, "max_adl_imb": MAX_ADL_IMB,
            "augmentierung": "1 Kopie, zufaellige Drehung, 50 % Spiegelung"},
    notes="Teil A Orientierung (Spiegelung/Drehung), Teil B Trainingsmatrix W/WK/L0/B2/B2r.")